# 03 · Pipeline walkthrough

Step-by-step figures of both pathways on one squeeze, as used for the thesis slides.
Figures are written to `outputs/figures/`.

* **Pathway 1:** YOLO-OBB detection → letter crops → ResNet18 classification → line reconstruction
* **Pathway 2:** YOLO-OBB detection → line grouping → rectified line strip → CRNN

In [ ]:
import os
from pathlib import Path

# Run from the repository root so the default config paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from trogs.config import load_config
from trogs.data.geometry import line_rect, warp_line
from trogs.inference.classifier_pipeline import ClassifierPipeline, ClassifierPipelineConfig
from trogs.inference.crnn_pipeline import CrnnPipeline, CrnnPipelineConfig, group_lines
from trogs.models.detector import detect, prediction_corners

IMG_PATH = "data/raw/test_fullsized_images/IAS_27288_Rotation1_300dpi.png"
FIG_DIR = Path("outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def show(img, title, name, size=(12, 12), **kwargs):
    plt.figure(figsize=size)
    plt.imshow(img, **kwargs)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=300, bbox_inches="tight")
    plt.show()


def obb_polygon(pred):
    """OBB corners from the SAHI mask (segmentation or polygon, depending on the SAHI version)."""
    mask = getattr(pred, "mask", None)
    if mask is not None and getattr(mask, "segmentation", None):
        return np.array(mask.segmentation[0], dtype=np.float32).reshape(-1, 2)
    return prediction_corners(pred)

## Pathway 1: detection + classification

In [ ]:
p1 = ClassifierPipeline(load_config(ClassifierPipelineConfig, "configs/pipeline/classifier.yaml"))
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
preds = detect(IMG_PATH, p1.detector, p1.slice_params)
print(f"Detected {len(preds)} letters")

show(img_rgb, "Input squeeze image", "pipeline_step1_input.png")
overlay = img_rgb.copy()
for pred in preds:
    cv2.polylines(overlay, [obb_polygon(pred).astype(np.int32)], True, (0, 255, 0), 4)
show(overlay, "YOLO-OBB letter detection", "pipeline_step2_detection.png")

In [ ]:
# Five letters spread over the image, with their top classifier prediction
samples = [preds[i] for i in np.linspace(0, len(preds) - 1, 5, dtype=int)]
fig, axes = plt.subplots(2, len(samples), figsize=(18, 8))
for i, pred in enumerate(samples):
    x1, y1, x2, y2 = map(int, pred.bbox.to_xyxy())
    crop = img_rgb[y1:y2, x1:x2]
    with torch.no_grad():
        probs = F.softmax(p1.classifier(p1.transform(Image.fromarray(crop)).unsqueeze(0).to(p1.device)), dim=1)
    conf, idx = torch.max(probs, 1)
    label = p1.class_names[idx.item()]
    axes[0, i].imshow(crop)
    axes[0, i].set_title(label)
    axes[0, i].axis("off")
    axes[1, i].bar([label], [conf.item()])
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_ylabel("Confidence")
plt.suptitle("ResNet18 letter classification")
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_step4_classification.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
transcript, boxes = p1.run(IMG_PATH)
print(transcript)

## Pathway 2: line grouping + CRNN

In [ ]:
p2 = CrnnPipeline(load_config(CrnnPipelineConfig, "configs/pipeline/crnn.yaml"))
lines = group_lines(detect(IMG_PATH, p2.detector, p2.slice_params))
print(f"{len(lines)} lines")

vis = img.copy()
for line in lines:
    box = cv2.boxPoints(cv2.minAreaRect(np.vstack([b["pts"] for b in line])))
    cv2.polylines(vis, [np.int32(box)], True, (0, 255, 0), 5)
show(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), "Step 1: line localisation", "crnn_step1_lines.png", size=(14, 14))

In [ ]:
# Longest line: rectified strip and its recognised text
line = max(lines, key=len)
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
center, w, h, angle = line_rect(np.vstack([b["pts"] for b in line]))
strip = warp_line(img_gray, center, w + h, h, angle, target_h=32, border_value=0)
show(strip, "Step 2: rectified text strip", "crnn_step2_strip.png", size=(16, 2), cmap="gray")
print("CRNN:", p2.recognise_line(img_gray, line))

In [ ]:
transcript, _, _ = p2.run(IMG_PATH)
print("\n".join(transcript))